#### Transformers — Encoder (BERT) vs Decoder (GPT) s

##### What a Transformer is
- Deep learning architecture that understands text using Self-Attention.
- Processes all words in parallel, not one at a time like RNN/LSTM.
- Learns relationships between every word and every other word at once.

Result: faster than RNN/LSTM, and better at long-range dependencies.

##### Sequential vs parallel processing
- RNN/LSTM: word 1 to word 2 to word 3, strictly in order, slow.
- Transformer: all words enter together, Self-Attention finds the relationships, fast.

##### Evolution of NLP models
- One-Hot / Bag of Words / TF-IDF: captures word presence, loses meaning, context, and order.
- Word2Vec: captures semantic meaning, but weak contextual understanding (same vector regardless of sentence).
- RNN / LSTM / BiLSTM: maintains sequence info, but slow due to step-by-step processing.
- Transformers (BERT, GPT, T5...): context + meaning + sequence + speed together. Most effective and scalable.

##### Three types of Transformer models
- BERT (Encoder-only): understanding text. FD example: classify an email, detect intent.
- GPT (Decoder-only): generating text. FD example: draft an FD reply email.
- T5 (Encoder-Decoder): summarization, translation, text-to-text. FD example: summarize + draft a reply together.


#### ENCODER (BERT)

##### What BERT is
- The Encoder half of the Transformer.
- Reads the entire sentence at once.
- Built for understanding text, not generating it.

##### Why BERT is bidirectional
- Looks left AND right around a word.
- Example: "My credit card [MASK] is overdue" → uses both sides to predict "payment."
- This is why BERT is strong at understanding.

##### BERT-base architecture, the 3 core numbers
- Hidden size = 768 → every token becomes a 768-dim vector; the "width" of the model.
- Layers = 12 → 12 stacked encoder blocks (self-attention + FFN each); the "depth" of the model.
- Attention heads = 12 → each head gets a 64-dim slice (768 ÷ 12); outputs concatenated back to 768.

##### BERT-base vs other popular models
- DistilBERT: 768 hidden, 6 layers, 12 heads, ~66M params.
- BERT-base: 768 hidden, 12 layers, 12 heads, ~110M params.
- GPT-2 Small: 768 hidden, 12 layers, 12 heads, ~117M params (same shape as BERT-base, but it's a decoder).
- BERT-large: 1024 hidden, 24 layers, 16 heads, ~340M params.
- GPT-2 Medium: 1024 hidden, 24 layers, 16 heads, ~345M params.
- MuRIL-base: 768 hidden, 12 layers, 12 heads, ~236M params.
- LLaMA-7B: 4096 hidden, 32 layers, 32 heads, ~7B params.
- GPT-4 / Claude 3: undisclosed.


##### Overall Steps in Encoders
Step 1: Input Embedding
- Goal example: classify "My credit card payment is overdue... Can you waive the penalty?"
- Why: neural nets only understand numbers; embeddings also capture meaning (credit ↔ payment ↔ penalty are related).
- Tokenization (WordPiece): breaks words into subwords, ~30K vocabulary. "payment overdue" → "pay" + "##ment" + "overdue."
- Token IDs: each piece maps to a unique vocabulary ID. ["pay","##ment","overdue"] → [3471, 2964, 7775].
- 768-D embedding: each ID becomes a dense 768-dim vector, learned in pretraining. 3471 → [0.12, -0.33, 0.45, ...].

Step 2: Positional + Segment Embedding
- Why positional: parallel processing has no built-in order; "Customer paid bank" ≠ "Bank paid customer" needs to be distinguishable.
BERT and all GPT versions: learned positional embeddings.
- Original 2017 Transformer: fixed sine/cosine positional encoding instead (close positions look similar, far positions look different, generalizes to longer sequences).
- Segment embedding: used when 2 sentences are fed together (e.g. NSP pretraining). Sentence A → segment 0, Sentence B → segment 1.
Single-sentence tasks (like email classification): every token gets segment ID 0.
- Final input = Token Embedding + Positional Embedding + Segment Embedding → one 768-D vector per token, already carrying meaning + position + sentence identity before Self-Attention even starts.

Step 3: Self-Attention
- Roles: Query (what this word looks for), Key (what this word offers), Value (the actual info carried).
- Worked example for "penalty": Query vs Keys → "credit card" = high relevance, "salary delayed" = medium, "payment overdue" = medium, "my" = very low.
- Softmax turns relevance into weights: credit card 45%, salary delayed 30%, payment overdue 20%, my 5%.
- Weighted sum of Values → final "penalty" representation = blend of those 4 meanings. This output is called a contextual embedding.

Interview point: Q and K decide where to attend; V decides what gets pulled in.

Formula: Attention(Q,K,V) = softmax(QKᵗ / √d_k) · V.

Why √d_k: d_k = 64 in BERT (768 ÷ 12 heads). Large dot-product scores → softmax too sharp → vanishing gradients. Scaling by √64 = 8 keeps scores balanced.

Without scaling: scores 7.5, 5.0, 0.5 → softmax 0.90, 0.09, 0.01 (too sharp).
With scaling: scores 0.94, 0.63, 0.06 → softmax 0.45, 0.30, 0.25 (balanced).

Step 4: Multi-Head Attention
- Multiple heads run in parallel, each learns a different relationship type simultaneously.
- BERT-base example heads: head 1 = payment status, head 2 = cause/effect, head 3 = customer request, head 4 = product/identity, heads 5–12 = other patterns.

Flow: input → multiple Q/K/V sets (one per head) → each head attends independently → outputs concatenated → linear layer combines into one rich representation.

Math: head_i = Attention(QW_i^Q, KW_i^K, VW_i^V); MultiHead = Concat(head_1...head_h)·Wᵒ.

Step 5: Add & Normalize
- Purpose: keep old info + new info together, and keep values numerically stable.
- Add (residual): output = input + attention-output. Original embedding isn't thrown away.
- Normalize (LayerNorm): rescales values to a stable range; prevents exploding/vanishing values.

Step 6: Feedforward Network (FFN)
- Refines each token individually: removes noise, strengthens useful patterns.
- Formula: FFN(x) = W₂ · GeLU(W₁x + b₁) + b₂.
- BERT-base: expands 768 → 3072 (4×), then shrinks back 3072 → 768.
- GeLU specifically chosen over ReLU — smoother, better for language tasks.

Step 7: Add & Normalize (again)
- Same idea as  13, applied after the FFN this time.
- Output = LayerNorm(input + FFN-output).
- Preserves contextual info + FFN refinements; helps deep models avoid information loss.

Step 8: Stacked Encoder Layers
- 12 identical blocks stacked (BERT-base); each block = attention → add&norm → FFN → add&norm.
    - Early layers (≈1–3): basic word/phrase relationships.
    - Middle layers (≈4–9): sentence meaning, context.
    - Deep layers (≈10–12): intent, reasoning, high-level semantics.
    - Net effect: surface-level text understanding → high-level semantic understanding.

Step 9: Special tokens [CLS] and [SEP]
- [CLS]: added at start of input; its final representation summarizes the whole input; used for classification tasks.
- [SEP]: marks sentence boundary / end of input.
- Example: [CLS] learns "credit card penalty-waiver request due to salary delay"; [SEP] marks end of sentence.

Step 10: Classification Head
- Added only during fine-tuning — not part of pretrained BERT.
- Final [CLS] vector (768-D) → Dense layer → Softmax → class probabilities.
- Example output: Credit Card Query = 0.96, Non-Credit Query = 0.04 → prediction = Credit Card Query.


##### BERT pretraining: Masked Language Modeling (MLM)
- Randomly masks words, predicts them using both left and right context.
- Example: "Customer requested waiver of [MASK] because salary was delayed" → correct = "penalty."
- Learns: "waiver" relates to "penalty/charges," "salary delayed" = financial-hardship context.
- Context-dependent meaning example: "bank" = financial institution / river side / banking org, depending on surrounding words.
- Key point: MLM is what makes BERT bidirectional, since it uses both directions to predict masked words.


##### BERT pretraining: Next Sentence Prediction (NSP)
- Given Sentence A + Sentence B, predicts whether B logically follows A.
- Related example: "Salary delayed, payment overdue" → "Customer requested penalty waiver" = logical continuation.
- Unrelated example: same Sentence A → "Customer booked movie tickets online" = unrelated.
- Learns: cause/effect, logical sentence flow, connected vs disconnected ideas.

##### Why MLM + NSP together
- MLM → contextual word meaning (bidirectional).
- NSP → sentence-relationship and logical flow.
- Together: semantic understanding + syntactic structure + contextual reasoning + sentence-level logic.
- FD example: MLM understands "penalty waiver," "interest rate," "FD renewal"; NSP understands "salary delayed" → "payment overdue" as connected.

##### ReLU vs GeLU (bank-balance analogy)
- Balance = input, spending power = output.
- ReLU (hard rule): 
    - negative balance → spending power = 0 exactly, no matter how negative. 
    - Positive balance → full spending power. −5000 → 0; −1 → 0; +5000 → 5000.
    - Problem: even −1 gets fully blocked — no middle ground.
- GeLU (smart rule): 
    - negative balance → mostly blocked, but a little signal still flows through. 
    - Positive balance → nearly full spending power. −5000 → ≈0; −1 → ≈−0.16 (small, non-zero); +5000 → ≈5000.
    - Advantage: small negatives aren't fully killed — some signal survives.
- One-line summary: ReLU = on/off switch, negative always 0, neurons can "die," used in old CNNs. GeLU = dimmer switch, negative ≈ (not exactly) 0, neurons keep learning, used in GPT/BERT/modern LLMs.